# CAWOT-CM coreset sweep (V0 + V1) — Kaggle

**Source of truth: HuggingFace** [`TruongVox/Cawot-dataset`](https://huggingface.co/datasets/TruongVox/Cawot-dataset).

Compares 4 selection methods across budgets {5,10,20,40}%:
- **random** — uniform baseline
- **v0** — farthest-from-centroid (diversity / atypical) [plan V0]
- **v0_proto** — closest-to-centroid (prototype / representative) [diagnostic]
- **v1** — cross-modal cost + facility location [plan V1, the method]

**Setup (Kaggle UI):** GPU P100 + Internet ON. No external dataset needed.

## 1. Env

In [ ]:
!nvidia-smi -L
!df -h /kaggle/working | tail -1

## 2. Clone repo + install deps

In [ ]:
import os
if not os.path.exists("/kaggle/working/cawot-cm"):
    !git clone https://github.com/HohoHocCode/cawot-cm.git /kaggle/working/cawot-cm
%cd /kaggle/working/cawot-cm
!pip install -q open_clip_torch faiss-gpu-cu12 einops huggingface_hub

## 3. Download N shards from HuggingFace

5 shards ≈ 7 GB download, ~14 GB extracted (~65K image-caption pairs).

In [ ]:
NUM_SHARDS = 5
!python scripts/setup_data.py --output /kaggle/working/pab_data --num-shards {NUM_SHARDS}

## 4. Sanity check that one image resolves

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/cawot-cm")
from src.data import build_pool, TrainPoolDataset
from torchvision import transforms
anns, shard_roots = build_pool("/kaggle/working/pab_data/annotations",
                               "/kaggle/working/pab_data/images", sample_size=100, seed=42)
print(f"loaded {len(anns)} annotations; shards: {sorted(shard_roots)}")
tx = transforms.Compose([transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor()])
s = TrainPoolDataset(anns, shard_roots, image_transform=tx)[0]
print(f"image {s['image'].shape}; caption: {s['caption'][:80]}...")
print("\u2713 images resolve")

## 5. Run the sweep (V0 family + V1)

Default config: 4 methods × 4 budgets × 1 seed = 16 fine-tune runs.
On P100:
- Extract image+text embeddings (50K, one pass): ~18-22 min (cached after)
- 16 fine-tune runs + 16 evals: ~2.5-3 h
- **Total ~3-3.5 h for 1 seed**

After confirming V1 beats Random, set `train.seeds: [42, 1, 2]` for error bars (note: 3 seeds ≈ 8-9 h, may need to split Kaggle sessions — or run 3 seeds only at the 20% headline).

In [ ]:
!python scripts/run_sweep.py --config config.yaml

## 6. Results table

In [ ]:
import json, pandas as pd
summary = json.load(open("/kaggle/working/outputs/eval/summary.json"))
print("zeroshot mean_R@1:", summary["zeroshot"]["mean_R@1"])
methods = [m for m in ["random", "v0", "v0_proto", "v1"] if m in summary]
budgets = sorted(float(b) for b in summary[methods[0]].keys())
tbl = {m: [summary[m][str(b)]["mean_R@1_mean"] for b in budgets] for m in methods}
df = pd.DataFrame(tbl, index=[f"{int(b*100)}%" for b in budgets])
df.index.name = "budget"
if "v1" in df and "random" in df:
    df["v1 - random"] = df["v1"] - df["random"]
df

## 7. Plot R@1 vs budget

In [ ]:
import matplotlib.pyplot as plt
markers = {"random": "o", "v0": "^", "v0_proto": "v", "v1": "s"}
x = [b * 100 for b in budgets]
plt.figure(figsize=(7.5, 5))
for m in methods:
    y = [summary[m][str(b)]["mean_R@1_mean"] for b in budgets]
    e = [summary[m][str(b)]["mean_R@1_std"] for b in budgets]
    plt.errorbar(x, y, yerr=e, marker=markers.get(m, "o"), capsize=3, label=m)
plt.axhline(summary["zeroshot"]["mean_R@1"], ls="--", c="gray",
            label=f"zero-shot ({summary['zeroshot']['mean_R@1']:.1f})")
plt.xlabel("Budget (% of train pool)"); plt.ylabel("mean R@1")
plt.title("Coreset selection: V0 family vs V1"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("/kaggle/working/outputs/eval/sweep_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. How to read (for the report)

Expected ordering at **low budget (5%)**:
`v1 ≳ v0_proto > random > v0` and `zero-shot` lowest.

- **v1 > random** at low budget = the headline V1 result (representative cross-modal coverage helps when data is scarce).
- **v0 < random** (farthest-point hurts) = reproduces the Sorscher 2022 insight: at low budget keep prototypical, not atypical, samples.
- **v0_proto > random** confirms the same insight from the other side.
- Gaps shrink toward **40%** as all methods converge (selection matters less when budget is loose).

Artifacts to keep for the report (Save Version → Save & Run All):
`outputs/eval/summary.json`, `outputs/eval/records.csv`, `outputs/eval/sweep_curve.png`.